# Variant 1 — TaskTransformer (complete) + TimeTransformer
Pipeline: token combinato (regione+task) → predice il prossimo token, poi il tempo.

In [ ]:
import os
import torch
import pandas as pd
from config import DATA_DIR

from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.objects.log.obj import Trace, Event, EventLog

from core import TaskTransformer, TimeTransformer
from core.training import train_task, train_time
from utils import get_decoding, get_encoding, hamming_distance, edit_distance_weighted_levenshtein

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
info = torch.load(DATA_DIR / 'prepared_data.pt', map_location=device, weights_only=False)
n = info['n']

data_task = {
    'train_complete': info['data_complete'][:n],
    'val_complete': info['data_complete'][n:],
}
data_time = {
    'train_complete': info['data_complete'][:n],
    'val_complete': info['data_complete'][n:],
    'train_times': info['data_times'][:n],
    'val_times': info['data_times'][n:],
}

vocab_size = info['vocab_size_complete']
num_regions = info['num_regions']
num_tasks = info['num_tasks']

decode_complete = lambda b: [info['id_to_bit_complete'][x] for x in b]
encode_complete = lambda a: [info['bit_to_id_complete'][tuple(x)] for x in a]

net = info['net']

print(f'vocab={vocab_size}, n_train={n}')

In [ ]:
param_task_region_transformer_default = dict(block_size=256, n_embd=256, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, weight_decay=0.01, batch_size=16, max_iters=500, eval_iters=200, eval_interval=150)
param_time_transformer_default = dict(block_size=64,  n_embd=128, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, weight_decay=0.01, batch_size=16, max_iters=500, eval_iters=200, eval_interval=150)

params = torch.load(DATA_DIR / 'v1_best_params.pt', map_location=device, weights_only=False) if os.path.exists(DATA_DIR / 'v1_best_params.pt') else None

p_task_region = param_task_region_transformer_default
p_time = param_time_transformer_default
if params is not None:
    p_task_region = {**params['TaskTransformer'], 'max_iters': 1500, 'eval_iters': 500, 'eval_interval': 500}
    p_time = {**params['TimeTransformer'],  'max_iters': 1500, 'eval_iters': 500, 'eval_interval': 500}

print('Task Region params:', p_task_region)
print('Time params:', p_time)

In [17]:
model_task = TaskTransformer(
    task_vocab_size=vocab_size,
    block_size=p_task_region['block_size'],
    n_embd=p_task_region['n_embd'],
    dropout=p_task_region['dropout'],
    n_head=p_task_region['n_head'],
    n_layer=p_task_region['n_layer'],
).to(device)

train_task(model_task, data_task, p_task_region, device, data_key='complete', printing=True)
print('TaskTransformer trained.')

step 0: train loss 3.5884, val loss 3.5904
step 500: train loss 0.7419, val loss 0.7985
step 1000: train loss 0.6702, val loss 0.8526
step 1500: train loss 0.5470, val loss 0.9422
step 2000: train loss 0.4266, val loss 1.0548
step 2500: train loss 0.3438, val loss 1.1596
step 3000: train loss 0.2922, val loss 1.2410
step 3500: train loss 0.2619, val loss 1.3143
step 4000: train loss 0.2405, val loss 1.3461
step 4500: train loss 0.2241, val loss 1.3937
step 5000: train loss 0.2163, val loss 1.4421
step 5500: train loss 0.2063, val loss 1.4838
step 6000: train loss 0.1991, val loss 1.5044
step 6500: train loss 0.1977, val loss 1.5182
step 7000: train loss 0.1909, val loss 1.5633
step 7500: train loss 0.1883, val loss 1.5795
step 8000: train loss 0.1855, val loss 1.5956
step 8500: train loss 0.1843, val loss 1.6116
step 9000: train loss 0.1811, val loss 1.6388
step 9500: train loss 0.1788, val loss 1.6543
step 9999: train loss 0.1775, val loss 1.6856
TaskTransformer trained.


In [18]:
model_time = TimeTransformer(
    vocab_size_task=vocab_size,
    block_size=p_time['block_size'],
    n_embd=p_time['n_embd'],
    dropout=p_time['dropout'],
    n_head=p_time['n_head'],
    n_layer=p_time['n_layer'],
    separated_regions=False,
).to(device)

train_time(model_time, data_time, p_time, device, printing=True)
print('TimeTransformer trained.')

step 0: train loss 0.1343, val loss 0.1282
step 500: train loss 0.0049, val loss 0.0049
step 1000: train loss 0.0043, val loss 0.0043
step 1500: train loss 0.0040, val loss 0.0040
step 2000: train loss 0.0038, val loss 0.0037
step 2500: train loss 0.0037, val loss 0.0036
step 3000: train loss 0.0037, val loss 0.0036
step 3500: train loss 0.0035, val loss 0.0036
step 4000: train loss 0.0035, val loss 0.0036
step 4500: train loss 0.0035, val loss 0.0035
step 5000: train loss 0.0034, val loss 0.0034
step 5500: train loss 0.0034, val loss 0.0033
step 6000: train loss 0.0036, val loss 0.0035
step 6500: train loss 0.0036, val loss 0.0035
step 7000: train loss 0.0035, val loss 0.0034
step 7500: train loss 0.0035, val loss 0.0035
step 8000: train loss 0.0035, val loss 0.0034
step 8500: train loss 0.0034, val loss 0.0034
step 9000: train loss 0.0034, val loss 0.0033
step 9500: train loss 0.0034, val loss 0.0034
step 9999: train loss 0.0034, val loss 0.0035
TimeTransformer trained.


In [19]:
max_new_tokens = 100

sep_id = info['bit_to_id_complete'][tuple([0]*(num_regions+num_tasks))]
mean_sep_delta = info['data_times'][:n][info['data_complete'][:n] == sep_id].float().mean().item()
print(mean_sep_delta)

context = torch.tensor(encode_complete([[0]*(num_regions+num_tasks)]), dtype=torch.long, device=device).unsqueeze(0)
context_time = torch.tensor([mean_sep_delta], dtype=torch.float32, device=device).unsqueeze(0)

generated_ids = []
generated_times = []

for _ in range(max_new_tokens):
    next_id = model_task.predict_next_task(idx_task=context, block_size=p_task_region['block_size'])
    context = torch.cat((context, next_id), dim=1)
    context_aligned = context[:, 1:]   # allineamento offset

    next_time = model_time.predict_next_time(idx_tasks=context_aligned, idx_times=context_time, block_size=p_time['block_size'])
    context_time = torch.cat((context_time, next_time), dim=1)

    generated_ids.append(next_id.item())
    generated_times.append(next_time.item())

decoded = decode_complete(generated_ids)
'''for i, (step, t) in enumerate(zip(decoded, generated_times)):
    print(f'{i:02d}: {[int(b) for b in step]} — time={round(t,4)}')'''

0.3985038101673126


"for i, (step, t) in enumerate(zip(decoded, generated_times)):\n    print(f'{i:02d}: {[int(b) for b in step]} — time={round(t,4)}')"

In [20]:
# Raggruppa la sequenza in tracce separate (separatore = vettore zero)
traces_generated, current = [], []
for step in decoded:
    bits = [int(b) for b in step]
    current.append(bits)
    if bits == [0]*(num_regions+num_tasks):
        if len(current) > 1:
            traces_generated.append(current)
        current = []

for i,trace in enumerate(traces_generated):
    print(f"{i}: {trace}")

0: [[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
1: [[1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [21]:
traces_decoded = get_decoding(traces_generated, net.regions, net.tasks)
print(traces_decoded)

[['start_T15', 'end_T15', 'start_T16', 'end_T16'], ['start_T1', 'start_T8', 'end_T8', 'start_T10', 'end_T10', 'end_T1', 'start_T2', 'end_T2', 'start_T11', 'end_T11'], ['start_T6', 'start_T8', 'end_T8', 'start_T10', 'end_T10', 'end_T6', 'start_T11', 'end_T11'], ['start_T13', 'start_T14', 'end_T14', 'end_T13'], ['start_T7', 'start_T9', 'end_T7', 'end_T9', 'start_T10', 'end_T10', 'start_T12', 'end_T12'], ['start_T8', 'end_T8', 'start_T10', 'end_T10', 'start_T1', 'end_T1', 'start_T9', 'end_T9', 'start_T10', 'start_T2', 'end_T10', 'end_T2', 'start_T12', 'end_T12'], ['start_T16', 'start_T15', 'end_T16', 'end_T15'], ['start_T16', 'start_T15', 'end_T16', 'end_T15'], ['start_T13', 'start_T14', 'end_T14', 'end_T13'], ['start_T8', 'end_T8', 'start_T7', 'start_T10', 'end_T10', 'end_T7', 'start_T11', 'end_T11'], ['start_T8', 'end_T8', 'start_T10', 'start_T7', 'end_T10', 'end_T7', 'start_T11', 'end_T11'], ['start_T3', 'start_T9', 'end_T3', 'start_T4', 'end_T4', 'end_T9', 'start_T5', 'start_T10', 'en

In [22]:
'''
PROBLEMA: Può generare tracce sfasate, con più eventi per step.
get_decoding non funziona sotto questo punto di vista. o meglio, funziona generando eventi in più (tipo 6 eventi da 4 step perchè ci sono 2 step generati male)
'''

classifier_dict_tasks = info['classifier_dict_tasks']
dict_task_step_encoding = info['dict_task_step_encoding']
current_trace_context = []
#traces_decoded_list = [step for trace in traces_decoded for step in trace]

tasks_previous = [0] * num_tasks
for i, (step, t) in enumerate(zip(decoded, generated_times)):
    bits = [int(b) for b in step]
    is_sep = bits == [0]*(num_regions+num_tasks)
    tasks_step = bits[num_regions:]

    step_events = []
    for j, task in enumerate(tasks_step):
        if task != tasks_previous[j]:
            step_events.append(("start_" if task == 1 else "end_") + net.tasks[j])
    tasks_previous = tasks_step

    if len(step_events) == 1:
        event_name = step_events[0]
        if event_name in classifier_dict_tasks:
            rt, max_len = classifier_dict_tasks[event_name]
            ctx = list(reversed(current_trace_context))[:max_len]
            padded = ctx + ['PAD'] * (max_len - len(ctx))
            encoded = [dict_task_step_encoding.get(s, dict_task_step_encoding['PAD']) for s in padded]
            expected = round(rt.predict([encoded])[0], 4)
        else: # Non ci dovrebbe mai entrare in teoria
            expected = 0.0 if not current_trace_context else "n/d"
        current_trace_context.append(event_name)
        note = ""
    elif len(step_events) == 0: # step che non genera eventi (es. cambia solo la regione)
        expected, note = "—", "(step senza evento)"
    else:# step malformato: accende/spegne 2 task insieme
        expected, note = "—", f"(step ambiguo: {step_events})"

    if is_sep:
        current_trace_context = []

    print(f'{i:02d}: {bits} — time={round(t,4)} | expected={expected} {note}')

00: [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.4443 | expected=0.0 
01: [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.7329 | expected=0.4286 
02: [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.9845 | expected=1.0 
03: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=1.0099 | expected=1.0 
04: [1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.0078 | expected=0.0 
05: [1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0] — time=1.0159 | expected=1.0 
06: [1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=1.0139 | expected=1.0 
07: [1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] — time=0.4285 | expected=0.4286 
08: [1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [23]:
# Allineamento con conformance checking pm4py
check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
loop = tuple(["back_L"]) + tuple(["end_L"])
silent_prefixes = start + end + loop

# Creo i parametri per l'allineamento
model_cost, sync_cost = {}, {}
for t in net.net.transitions: # Prendo tutte le transizioni
    if t.label is None or (t.label is not None and t.label.startswith(silent_prefixes)): # Se è una transizione silente
        model_cost[t] = 0
        sync_cost[t] = 10000
    else: # Se è un task vero e proprio
        model_cost[t] = 10000
        sync_cost[t] = 0

alignment_params = {
    alignments.Parameters.PARAM_MODEL_COST_FUNCTION: model_cost,
    alignments.Parameters.PARAM_SYNC_COST_FUNCTION: sync_cost,
}

# Creiamo l'EventLog di ogni traccia per poi poterla allineare
eventlog_traces = EventLog()
for trace in traces_decoded:
    t = Trace()
    for activity in trace:
        t.append(Event({'concept:name': activity}))
    eventlog_traces.append(t)

aligned_traces = alignments.apply(eventlog_traces, net.net, net.initial_marking, net.final_marking, parameters=alignment_params)
for i,trace in enumerate(aligned_traces):
    print(f"{i}: {trace}")

aligning log, completed variants ::   0%|          | 0/10 [00:00<?, ?it/s]

0: {'alignment': [('>>', 'start_X0'), ('>>', 'start_P10'), ('start_T15', 'start_T15'), ('end_T15', 'end_T15'), ('start_T16', 'start_T16'), ('end_T16', 'end_T16'), ('>>', 'end_P10'), ('>>', 'end_X0')], 'cost': 0, 'visited_states': 8, 'queued_states': 22, 'traversed_arcs': 22, 'lp_solved': 1, 'fitness': 1.0, 'bwc': 80000}
1: {'alignment': [('>>', 'start_X0'), ('>>', 'start_P2'), ('>>', 'start_X3'), ('start_T1', 'start_T1'), ('>>', 'start_X7'), ('start_T8', 'start_T8'), ('end_T8', 'end_T8'), ('>>', 'end_X7'), ('start_T10', 'start_T10'), ('end_T10', 'end_T10'), ('end_T1', 'end_T1'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('>>', 'end_X3'), ('>>', 'end_P2'), ('>>', 'start_X8'), ('start_T11', 'start_T11'), ('end_T11', 'end_T11'), ('>>', 'end_X8'), ('>>', 'end_X0')], 'cost': 0, 'visited_states': 22, 'queued_states': 75, 'traversed_arcs': 75, 'lp_solved': 1, 'fitness': 1.0, 'bwc': 140000}
2: {'alignment': [('>>', 'start_X0'), ('>>', 'start_P2'), ('>>', 'start_X3'), ('start_T6', 'start_

In [24]:
'''Codifichiamo le tracce allineate (per poi poterle confrontare con quelle generate dal transformer)'''

silent_prefixes = start + end + tuple(["back_L"])

aligned_traceEncoded_regions, aligned_traceEncoded_tasks = get_encoding(
    [[step for _, step in a['alignment'] if step and not step.startswith(silent_prefixes) and step != '>>']
     for a in aligned_traces],
    net.regions, net.tasks, net.open_clauses, net.end_clauses
)

print(aligned_traceEncoded_regions)

df_aligned_traces = pd.concat([aligned_traceEncoded_regions, aligned_traceEncoded_tasks], axis=0)

df_aligned_traces

     0   1   2   3   4   5   6   7   8   9   ...  78  79  80  81  82  83  84  \
R0    1   1   1   0   1   1   1   1   1   1  ...   1   1   1   1   1   0   1   
R1    0   0   0   0   1   1   1   1   1   1  ...   1   1   1   1   1   0   0   
R10   1   1   1   0   0   0   0   0   0   0  ...   0   0   0   0   0   0   0   
R2    0   0   0   0   1   1   1   1   1   1  ...   1   1   1   0   0   0   0   
R3    0   0   0   0   1   1   1   1   1   1  ...   1   1   0   0   0   0   0   
R4    0   0   0   0   1   1   1   1   1   1  ...   0   0   0   0   0   0   0   
R5    0   0   0   0   0   0   0   0   0   0  ...   1   1   0   0   0   0   0   
R6    0   0   0   0   0   1   1   1   0   0  ...   1   1   1   0   0   0   0   
R7    0   0   0   0   0   1   0   0   0   0  ...   0   0   0   0   0   0   0   
R8    0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   1   0   0   
R9    0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   0   1   

     85  86  87  
R0    1   1   0  
R1 

,0,1,2,3,4,5,6,7,8,9,...,78,79,80,81,82,83,84,85,86,87
R0,1,1,1,0,1,1,1,1,1,1,...,1,1,1,1,1,0,1,1,1,0
R1,0,0,0,0,1,1,1,1,1,1,...,1,1,1,1,1,0,0,0,0,0
R10,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
R2,0,0,0,0,1,1,1,1,1,1,...,1,1,1,0,0,0,0,0,0,0
R3,0,0,0,0,1,1,1,1,1,1,...,1,1,0,0,0,0,0,0,0,0
R4,0,0,0,0,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
R5,0,0,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
R6,0,0,0,0,0,1,1,1,0,0,...,1,1,1,0,0,0,0,0,0,0
R7,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
R8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [25]:
'''Creo una lista delle tracce codificate (ogni traccia è una lista dove ogni elemento è una colonna del df, ossia uno step) --> più facili da confrontare quando calcoliamo la distanza'''

aligned_traces_encoded = []
aligned_trace_encoded = []
for element in df_aligned_traces.T.values:
    element = element.tolist()
    aligned_trace_encoded.append(element)
    if element == [0] * (num_regions+num_tasks):
        aligned_traces_encoded.append(aligned_trace_encoded)
        aligned_trace_encoded = []

# Andiamo a calcolare il costo con la edit distance (weighted_levenshtein)
costs = []
for i, (gen, aln) in enumerate(zip(traces_generated, aligned_traces_encoded)):
    cost = edit_distance_weighted_levenshtein(gen, aln, num_regions+num_tasks, num_regions+num_tasks, hamming_distance)
    costs.append(cost)
    print(f'Traccia {i}: edit_distance={cost}')

print(f'\nEdit distance media: {sum(costs)/len(costs):.2f}')

Traccia 0: edit_distance=0.0
Traccia 1: edit_distance=0.0
Traccia 2: edit_distance=0.0
Traccia 3: edit_distance=0.0
Traccia 4: edit_distance=0.0
Traccia 5: edit_distance=110.0
Traccia 6: edit_distance=0.0
Traccia 7: edit_distance=0.0
Traccia 8: edit_distance=0.0
Traccia 9: edit_distance=0.0
Traccia 10: edit_distance=0.0
Traccia 11: edit_distance=0.0
Traccia 12: edit_distance=0.0

Edit distance media: 8.46
